In [1]:
import math
import sys
import time


In [2]:
# Graph definition
graph = {
    'Indira Gandhi Airport': {'India Gate': 15, 'Qutub Minar': 20},
    'India Gate': {'Indira Gandhi Airport': 15, 'Red Fort': 10, 'Lotus Temple': 15, 'Humayun\'s Tomb': 10},
    'Red Fort': {'India Gate': 10, 'Lotus Temple': 30, 'Jama Masjid': 5},
    'Qutub Minar': {'Indira Gandhi Airport': 20, 'Lotus Temple': 15, 'Humayun\'s Tomb': 20},
    'Lotus Temple': {'India Gate': 15, 'Red Fort': 30, 'Qutub Minar': 15, 'Humayun\'s Tomb': 10, 'Akshardham Temple': 25},
    'Jama Masjid': {'Red Fort': 5, 'Akshardham Temple': 20},
    'Humayun\'s Tomb': {'India Gate': 10, 'Qutub Minar': 20, 'Lotus Temple': 10, 'Akshardham Temple': 15},
    'Akshardham Temple': {'Lotus Temple': 25, 'Humayun\'s Tomb': 15, 'Jama Masjid': 20}
}

# All unique locations in the graph (sorted for consistent processing)
all_locations = sorted(list(graph.keys()))
num_locations = len(all_locations)


In [3]:
# Pre-calculate minimum outgoing edge cost for each node
# This is used in the heuristic function.
min_costs_per_node = {}
for node, neighbors in graph.items():
    if neighbors:
        min_costs_per_node[node] = min(neighbors.values())
    else:
        min_costs_per_node[node] = 0 # Should not happen in a connected graph with nodes to visit

# Heuristic function for TSP-like problem (visiting all nodes)
# This heuristic calculates the sum of the minimum outgoing edge costs for all unvisited nodes.
# It is admissible because each unvisited node must eventually be part of the path,
# and traversing at least one edge from it is necessary.
def heuristic(current_node, visited_set):
    h_val = 0
    unvisited_nodes = [node for node in all_locations if node not in visited_set]

    # If all nodes are visited, the heuristic cost to the goal is 0
    if not unvisited_nodes:
        return 0

    # Sum of minimum outgoing edge costs for all remaining unvisited nodes.
    # This provides a lower bound on the cost to connect these nodes.
    for node in unvisited_nodes:
        h_val += min_costs_per_node.get(node, 0)

    return h_val

In [4]:
# Global variables for complexity tracking
nodes_expanded = 0
max_recursion_depth = 0
current_recursion_depth = 0

def rbfs(graph, current_node, path, visited_set, g, f_limit):
    """
    Recursive Best-First Search implementation to find a path that visits all nodes.

    Args:
        graph (dict): The graph representation.
        current_node (str): The current node in the search.
        path (list): The list of nodes visited so far in the current path.
        visited_set (frozenset): A set of all unique nodes visited so far.
        g (int): The cost from the start node to the current_node.
        f_limit (float): The current f-value limit for this branch.

    Returns:
        tuple: (f_value, path, new_f_limit_from_child)
               - f_value: The f-value of the best path found or the cutoff value.
               - path: The optimal path if found, otherwise None.
               - new_f_limit_from_child: The f-value that caused a cutoff, used for updating successors.
    """
    global nodes_expanded, max_recursion_depth, current_recursion_depth

    nodes_expanded += 1
    current_recursion_depth += 1
    max_recursion_depth = max(max_recursion_depth, current_recursion_depth)

    # Goal test: if all unique locations have been visited
    if len(visited_set) == num_locations:
        current_recursion_depth -= 1
        return g, path, None # Return cost, path, and None (no f_limit update needed)

    # Calculate f-value for the current node
    h_val = heuristic(current_node, visited_set)
    f = g + h_val

    # If the current f-value exceeds the f_limit, prune this branch
    if f > f_limit:
        current_recursion_depth -= 1
        return f, None, None # Return current f-value as cutoff, no path, no f_limit update

    successors = []
    for neighbor, cost in graph[current_node].items():
        new_g = g + cost
        new_path = path + [neighbor]
        new_visited_set = visited_set.union({neighbor}) # Create a new set for the successor

        # Calculate f-value for successor
        successor_h = heuristic(neighbor, new_visited_set)
        successor_f = new_g + successor_h
        successors.append((successor_f, new_g, neighbor, new_path, new_visited_set))

    if not successors:
        current_recursion_depth -= 1
        return float('inf'), None, None # No successors, path is a dead end

    # Sort successors by their f-values (f = g + h)
    successors.sort(key=lambda x: x[0])

    while True:
        best_f, best_g, best_node, best_path, best_visited_set = successors[0]

        # If the f-value of the best successor is greater than the current f_limit,
        # it means this branch cannot produce a better solution within the current limit.
        if best_f > f_limit:
            current_recursion_depth -= 1
            return best_f, None, None

        # Determine the f_limit for the recursive call to the best successor.
        # This is the minimum of the current f_limit and the f-value of the second-best successor.
        alternative_f_limit = successors[1][0] if len(successors) > 1 else float('inf')

        # Recursive call to explore the best successor
        result_f, result_path, new_f_limit_from_child = rbfs(graph, best_node, best_path, best_visited_set, best_g, min(f_limit, alternative_f_limit))

        if result_path is not None: # A solution was found in the recursive call
            current_recursion_depth -= 1
            return result_f, result_path, None # Pass the solution up

        # If no solution was found (cutoff occurred), update the f-value of the best successor
        # with the cutoff value returned by the child, and re-sort the successors list.
        # This ensures that the next iteration considers the updated priority.
        successors[0] = (result_f, best_g, best_node, best_path, best_visited_set)
        successors.sort(key=lambda x: x[0])

    # This line should ideally not be reached if the logic is sound and a solution exists.
    current_recursion_depth -= 1

def solve_tsp_rbfs(start_node, graph, all_locations):
    """
    Initializes and runs the RBFS algorithm for the TSP-like problem.

    Args:
        start_node (str): The starting location for the tour.
        graph (dict): The graph representation.
        all_locations (list): A list of all unique locations to be visited.

    Returns:
        tuple: (final_cost, final_path, expanded_nodes_count, max_depth, time_taken)
    """
    global nodes_expanded, max_recursion_depth, current_recursion_depth
    nodes_expanded = 0
    max_recursion_depth = 0
    current_recursion_depth = 0

    start_path = [start_node]
    start_visited_set = frozenset([start_node]) # Use frozenset for immutability and hashability
    start_g = 0

    # Initial f_limit for the top-level call is infinity
    initial_f_limit = float('inf')

    start_time = time.time()
    final_cost, final_path, _ = rbfs(graph, start_node, start_path, start_visited_set, start_g, initial_f_limit)
    end_time = time.time()

    time_taken = end_time - start_time

    return final_cost, final_path, nodes_expanded, max_recursion_depth, time_taken

# --- Execution ---
start_location = 'Indira Gandhi Airport'
final_cost, final_path, expanded_nodes_count, max_depth, exec_time = solve_tsp_rbfs(start_location, graph, all_locations)

print("--- Tour Agent Results (using Recursive Best First Search) ---")
print(f"1. Optimal Path: {final_path}")
print(f"2. Total Cost for the path: {final_cost}")
print(f"3. Number of squares (nodes) in the path: {len(final_path)}")
print("\n--- Complexity Analysis ---")
print(f"4. Time Complexity (proxy: nodes expanded): {expanded_nodes_count}")
print(f"5. Space Complexity (proxy: maximum recursion depth): {max_depth}")
print(f"   Execution Time: {exec_time:.6f} seconds")

--- Tour Agent Results (using Recursive Best First Search) ---
1. Optimal Path: ['Indira Gandhi Airport', 'Qutub Minar', 'Lotus Temple', "Humayun's Tomb", 'India Gate', 'Red Fort', 'Jama Masjid', 'Akshardham Temple']
2. Total Cost for the path: 90
3. Number of squares (nodes) in the path: 8

--- Complexity Analysis ---
4. Time Complexity (proxy: nodes expanded): 81
5. Space Complexity (proxy: maximum recursion depth): 8
   Execution Time: 0.000689 seconds
